# DistilBERT AML Memo Classifier - Colab GPU + Hydra

This notebook fine-tunes `distilbert-base-uncased` on `memo_dataset.csv` using Hydra config files created inside Colab.

Before running: in Colab, choose **Runtime > Change runtime type > T4 GPU**.
The setup cell checks CUDA and the Hydra config enables mixed precision automatically when a GPU is available.


In [ ]:
# Colab setup
!pip -q install transformers datasets scikit-learn mlflow accelerate hydra-core omegaconf

import json
import os
import random
from pathlib import Path

import hydra
import mlflow
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from omegaconf import DictConfig, OmegaConf
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_recall_curve,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. In Colab, use Runtime > Change runtime type > T4 GPU.")


In [ ]:
# Hydra configuration
# This creates a small conf/ tree in the Colab runtime, then composes it with Hydra.
conf_dir = Path('/content/conf')
(conf_dir / 'model').mkdir(parents=True, exist_ok=True)
(conf_dir / 'data').mkdir(parents=True, exist_ok=True)
(conf_dir / 'training').mkdir(parents=True, exist_ok=True)

(conf_dir / 'config_distilbert.yaml').write_text('defaults:\n  - model: distilbert\n  - data: memo\n  - training: distilbert_default\n  - _self_\n\nexperiment_name: distilbert_memo\n')

(conf_dir / 'model' / 'distilbert.yaml').write_text('# @package _global_\nmodel:\n  name: distilbert\n  model_name: distilbert-base-uncased\n  max_length: 64\n  num_labels: 2\n\nmetrics:\n  target_threshold: 0.80\n  primary_key: auc_pr\n')

(conf_dir / 'data' / 'memo.yaml').write_text('# Synthetic memo text dataset for DistilBERT training\ncsv_path: /content/memo_dataset.csv\ntest_size: 0.15\nseed: 42\n\n# Set to a number for smoke tests, or null for the full dataset.\nmax_samples: null\n')

(conf_dir / 'training' / 'distilbert_default.yaml').write_text('# Default DistilBERT fine-tuning settings\nepochs: 3\ntrain_batch_size: 32\neval_batch_size: 64\nlearning_rate: 2e-5\nweight_decay: 0.01\nwarmup_ratio: 0.1\nlogging_steps: 50\nfp16: ${gpu_available:false}\n\n# MLflow\nmlflow_experiment: aml_distilbert_memo\n\n# Output paths\noutput_dir: /content/checkpoints/distilbert\nmodel_subdir: memo_model\nmetrics_output: /content/distilbert_metrics.json\n')

(conf_dir / 'training' / 'distilbert_fast.yaml').write_text('# Fast smoke-test settings\nepochs: 1\ntrain_batch_size: 8\neval_batch_size: 16\nlearning_rate: 2e-5\nweight_decay: 0.01\nwarmup_ratio: 0.1\nlogging_steps: 10\nfp16: ${gpu_available:false}\nmax_samples: 500\n\nmlflow_experiment: aml_distilbert_memo\noutput_dir: /content/checkpoints/distilbert_fast\nmodel_subdir: memo_model\nmetrics_output: /content/distilbert_metrics.json\n')

OmegaConf.clear_resolvers()
OmegaConf.register_new_resolver('gpu_available', lambda default=False: torch.cuda.is_available())

# Change overrides here if needed. Examples:
# overrides = ['training=distilbert_fast']
# overrides = ['training.epochs=1', 'data.max_samples=5000']
overrides = []

with hydra.initialize_config_dir(config_dir=str(conf_dir), version_base='1.3'):
    cfg = hydra.compose(config_name='config_distilbert', overrides=overrides)

cfg = OmegaConf.create(OmegaConf.to_container(cfg, resolve=True))
Path(cfg.training.output_dir).mkdir(parents=True, exist_ok=True)
print(OmegaConf.to_yaml(cfg))


In [ ]:
# Upload data if it is not already in /content
csv_path = Path(cfg.data.csv_path)
if not csv_path.exists():
    try:
        from google.colab import files

        print('Upload memo_dataset.csv')
        uploaded = files.upload()
        if 'memo_dataset.csv' not in uploaded:
            raise FileNotFoundError('Please upload a file named memo_dataset.csv')
    except ModuleNotFoundError:
        raise FileNotFoundError(f'Dataset not found: {cfg.data.csv_path}')

print(f'Using dataset: {cfg.data.csv_path}')


In [ ]:
# Reproducibility
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(cfg.data.seed)


In [ ]:
# Load and preprocess
def load_data(csv_path: str, max_samples: int | None = None) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df['memo_text'] = df['memo_text'].fillna('').astype(str)
    df['memo_text'] = df['memo_text'].replace('', '[NO_MEMO]')

    if max_samples:
        df = df.sample(n=min(max_samples, len(df)), random_state=cfg.data.seed)

    print(
        f"Loaded {len(df):,} rows | "
        f"illicit: {df['is_illicit'].sum():,} ({df['is_illicit'].mean() * 100:.1f}%)"
    )
    return df


def tokenize(batch, tokenizer):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=cfg.model.max_length,
    )


In [ ]:
# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)

    auc_pr = average_precision_score(labels, probs)
    precision, recall, _ = precision_recall_curve(labels, probs)
    idx = np.searchsorted(recall[::-1], 0.8)
    prec_at_r80 = float(precision[::-1][idx]) if idx < len(precision) else 0.0
    report = classification_report(labels, preds, output_dict=True, zero_division=0)

    return {
        "auc_pr": round(float(auc_pr), 4),
        "prec_at_recall_80": round(prec_at_r80, 4),
        "f1_illicit": round(report.get("1", {}).get("f1-score", 0), 4),
        "precision_illicit": round(report.get("1", {}).get("precision", 0), 4),
        "recall_illicit": round(report.get("1", {}).get("recall", 0), 4),
        "accuracy": round(report.get("accuracy", 0), 4),
    }


In [ ]:
# Weighted Trainer for imbalanced AML labels
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fn = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


In [ ]:
# Training
def get_max_samples(config: DictConfig) -> int | None:
    return config.training.get('max_samples', None) or config.data.get('max_samples', None)


def train(config: DictConfig):
    df = load_data(config.data.csv_path, get_max_samples(config))
    train_df, eval_df = train_test_split(
        df,
        test_size=config.data.test_size,
        stratify=df['is_illicit'],
        random_state=config.data.seed,
    )
    print(f"Train: {len(train_df):,} | Eval: {len(eval_df):,}")

    tokenizer = AutoTokenizer.from_pretrained(config.model.model_name)

    def to_hf_dataset(df_split):
        return Dataset.from_dict(
            {
                'text': df_split['memo_text'].tolist(),
                'label': df_split['is_illicit'].astype(int).tolist(),
            }
        ).map(lambda batch: tokenize(batch, tokenizer), batched=True)

    train_ds = to_hf_dataset(train_df)
    eval_ds = to_hf_dataset(eval_df)

    model = AutoModelForSequenceClassification.from_pretrained(
        config.model.model_name,
        num_labels=config.model.num_labels,
    )

    n_pos = int(train_df['is_illicit'].sum())
    n_neg = len(train_df) - n_pos
    class_weights = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float32)
    print(f"Positive class weight: {class_weights[1].item():.1f}x")

    training_args = TrainingArguments(
        output_dir=config.training.output_dir,
        num_train_epochs=config.training.epochs,
        per_device_train_batch_size=config.training.train_batch_size,
        per_device_eval_batch_size=config.training.eval_batch_size,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model=config.metrics.primary_key,
        greater_is_better=True,
        learning_rate=config.training.learning_rate,
        weight_decay=config.training.weight_decay,
        warmup_ratio=config.training.warmup_ratio,
        logging_steps=config.training.logging_steps,
        fp16=config.training.fp16,
        report_to='none',
        seed=config.data.seed,
    )

    mlflow.set_experiment(config.training.mlflow_experiment)
    with mlflow.start_run(run_name=f"{config.experiment_name}_ep{config.training.epochs}"):
        mlflow.log_params(
            {
                k: v
                for k, v in OmegaConf.to_container(config, resolve=True).items()
                if isinstance(v, (str, int, float, bool))
            }
        )
        mlflow.log_params(
            {
                'model_name': config.model.model_name,
                'max_length': config.model.max_length,
                'train_rows': len(train_df),
                'eval_rows': len(eval_df),
                'pos_weight': round(class_weights[1].item(), 2),
            }
        )

        trainer = WeightedTrainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            processing_class=tokenizer,
            compute_metrics=compute_metrics,
            class_weights=class_weights,
        )

        trainer.train()
        eval_results = trainer.evaluate()

        print('\nFinal eval results:')
        for key, value in eval_results.items():
            if not key.startswith('eval_runtime'):
                print(f'  {key}: {value}')

        mlflow.log_metrics(
            {
                key.replace('eval_', ''): value
                for key, value in eval_results.items()
                if isinstance(value, (int, float))
            }
        )

        model_save_path = Path(config.training.output_dir) / config.training.model_subdir
        trainer.save_model(str(model_save_path))
        tokenizer.save_pretrained(str(model_save_path))
        print(f'\nModel saved to: {model_save_path}')

        metrics = {
            key: value
            for key, value in eval_results.items()
            if isinstance(value, float) and not key.startswith('eval_runtime')
        }
        with open(config.training.metrics_output, 'w') as f:
            json.dump(metrics, f, indent=2)
        print(f'Metrics saved to: {config.training.metrics_output}')

        mlflow.log_artifact(config.training.metrics_output)
        mlflow.log_artifacts(str(model_save_path), artifact_path=config.training.model_subdir)

    return eval_results


In [ ]:
# Run fine-tuning on GPU
results = train(cfg)


In [ ]:
# Download trained artifacts from Colab
from google.colab import files

!zip -qr /content/distilbert_memo_model.zip {cfg.training.output_dir}
files.download('/content/distilbert_memo_model.zip')
files.download(cfg.training.metrics_output)
